<a href="https://colab.research.google.com/github/NOA11235/ResNet-Computer-Vision/blob/main/ResNet_Computer_Vision_Model_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Noa Rosenbom, ID 330910415
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

# Data Preparation
transform = transforms.Compose([
    transforms.Resize((224, 224)),      # ResNet requires images
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # Normalization
                         std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 dataset with the new transforms
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

# Create DataLoader to feed data in batches
trainloader = torch.utils.data.DataLoader(trainset, batch_size=16,
                                          shuffle=True, num_workers=0)

# Model Setup
# Load pre-trained ResNet18 model
resnet_model = models.resnet18(pretrained=True)

for param in resnet_model.parameters():
    param.requires_grad = False

# Replace the last fully connected layer with a new one for 10 classes
# The new layer is created with requires_grad=True by default
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, 10)

# Training Setup
# Check for GPU availability
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# Move model to the computation device
resnet_model = resnet_model.to(device)

# Define Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer setup
# We only pass 'resnet_model.fc.parameters()' to the optimizer.
# This ensures only the new layer is trained.
optimizer = optim.SGD(resnet_model.fc.parameters(), lr=0.001, momentum=0.9)

# Training Loop
print("Starting training process...")

# Train for 1 epoch
for epoch in range(1):
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        # Get the inputs and labels
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass: compute predicted outputs by passing inputs to the model
        outputs = resnet_model(inputs)

        # Calculate loss
        loss = criterion(outputs, labels)

        #compute gradient of the loss with respect to model parameters
        loss.backward()

        # Optimizer step: perform a parameter update
        optimizer.step()

        # Print statistics
        running_loss += loss.item()
        if i % 100 == 99:    # Print every 100 mini-batches
            print(f'[Epoch {epoch + 1}, Batch {i + 1}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

print('Finished Training')

In [1]:
# Evaluate the Model on Test Data
# Test the model on images it hasn’t seen
# 'train=False' loads the test set, 10,000 new images

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

testloader = torch.utils.data.DataLoader(testset, batch_size=16,
                                         shuffle=False, num_workers=0)

# Set model to evaluation mode
# This tells the model we are testing, not training
resnet_model.eval()

correct = 0
total = 0

#  Validation Loop
# 'torch.no_grad()' turns off the gradient calculation engine.
# It makes the code run faster and use less memory since we don't need backprop.
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # Move data
        images, labels = images.to(device), labels.to(device)

        # Calculate outputs by running images through the network
        outputs = resnet_model(images)

        # The output is a list of 10 numbers
        # We want the index of the highest score.
        _, predicted = torch.max(outputs.data, 1)

        # Update counters
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

#  Calculate and Print Accuracy
accuracy = 100 * correct / total
print(f'Accuracy of the network on the 10000 test images: {accuracy:.2f}%')

# Check if we met the requirement
if accuracy >= 70:
    print("Success! We reached the target accuracy.")
else:
    print("Not enough yet. You might need to run the training loop for another epoch.")

NameError: name 'torchvision' is not defined

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F


#imshow_unorm: Helper Function that un-normalize and display image
def imshow_unorm(tensor_img, title):
    tensor_img = tensor_img.cpu().clone()


    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    # Reverse the formula: image = (image * std) + mean
    tensor_img = tensor_img * std + mean

    # Clamp values
    tensor_img = torch.clamp(tensor_img, 0, 1)

    # Convert values
    np_img = tensor_img.numpy().transpose((1, 2, 0))

    plt.imshow(np_img)
    plt.title(title, fontsize=8)
    plt.axis('off')

# Find Top 10  Errors

print("Searching for the top 10 worst mistakes...")

mistakes = []
class_names = ['Plane', 'Car', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# Ensure variables are available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
resnet_model = resnet_model.to(device)

# Using 'testloader' again to go through images
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        # Get raw output
        outputs = resnet_model(images)

        # Convert raw output
        probs = F.softmax(outputs, dim=1)

        # Get the highest probability and the class index
        max_prob, predicted = torch.max(probs, 1)

        # Check for mistakes inside this batch
        for i in range(len(labels)):
            if predicted[i] != labels[i]:

                mistakes.append({
                    'confidence': max_prob[i].item(),
                    'image': images[i],
                    'pred': class_names[predicted[i]],
                    'true': class_names[labels[i]]
                })

# Sort the mistakes list by
mistakes.sort(key=lambda x: x['confidence'], reverse=True)

# Take the top 10
top_10_mistakes = mistakes[:10]

#  Plot the Results
fig = plt.figure(figsize=(15, 6))
print(f"Found {len(mistakes)} total errors. Showing the top 10:")

for i, error in enumerate(top_10_mistakes):
    ax = fig.add_subplot(2, 5, i + 1)

    title_text = f"Pred: {error['pred']} ({error['confidence']*100:.1f}%)\nReal: {error['true']}"

    imshow_unorm(error['image'], title_text)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn

#  Define the Residual Block Class
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()

        #  Convert goes to BN goes to ReLU goes to Conv goes to BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut path
        # If the input shape changes,e need to adjust the shortcut 'x' to match the output size.

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        # 1. Pass through the main layers
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # 2. Add the shortcut
        out += self.shortcut(x)


        out = self.relu(out)
        return out

# Define the Full Model
class MyCustomResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(MyCustomResNet, self).__init__()

        # Initial preparation layer
        # Taking the input  and expanding to 64 features
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        #  The 3 Residual Blocks:
        # Each block doubles the channels and halves the size

        # Block 1: 64 to 128 channels
        self.layer1 = ResidualBlock(in_channels=64, out_channels=128, stride=2)

        # Block 2: 128 to 256 channels
        self.layer2 = ResidualBlock(in_channels=128, out_channels=256, stride=2)

        # Block 3: 256 to 512 channels
        self.layer3 = ResidualBlock(in_channels=256, out_channels=512, stride=2)

        #  Adaptive Pooling and Classifier
        # This forces the output to be 1x1 spatial size regardless of input
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Final fully connected layer
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        # Initial layers
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # Pass through the 3 residual blocks
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)


        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x





In [ ]:
import torch.optim as optim

#Initialize the Custom Model
# Create an instance of the new model we defined
my_model = MyCustomResNet(num_classes=10)

# Move to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training Custom Model on: {device}")
my_model = my_model.to(device)

#Setup Loss and Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(my_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

# Step 3: Training Loop
num_epochs = 12  # Give it enough time to learn
target_accuracy = 70.0

print("Starting training from scratch...")

for epoch in range(num_epochs):
    my_model.train()  # Set mode to training
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward + Backward + Optimize
        outputs = my_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Step the scheduler (optional)
    scheduler.step()

    # -----------------------------------------------------
    # Validation Check after each epoch
    # -----------------------------------------------------
    my_model.eval()  # Set mode to evaluation
    correct = 0
    total = 0

    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = my_model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = running_loss / len(trainloader)

    print(f"Epoch [{epoch+1}/{num_epochs}] -> Loss: {avg_loss:.4f} | Test Accuracy: {accuracy:.2f}%")

    # Check if we reached the goal
    if accuracy >= target_accuracy:
        print(f"\nSUCCESS! Reached {accuracy:.2f}% accuracy (Target: {target_accuracy}%)")
        print("Stopping training early.")
        break

print("Finished Training Process.")

In [ ]:
#Qwestion 1
import torch
import torch.nn as nn

class MyConv2d(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, kernel_size=(1,1), stride=1, padding=0):
        super().__init__()

        # Saving the arguments for the forward pass
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size # Expecting a tuple (height, width)
        self.stride = stride
        self.padding = padding

        # Initializing the weights
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size[0], kernel_size[1]))

        # Note: We didn't ask for Bias, so I'm skipping it

    def forward(self, x):
        # Input shape
        batch_size, in_c, h_in, w_in = x.shape

        k_h, k_w = self.kernel_size

        # Manual Padding:
        # If padding is required  we need to create a larger canvas
        if self.padding > 0:
            # Calculate new dimensions:
            padded_h = h_in + 2 * self.padding
            padded_w = w_in + 2 * self.padding

            # Create a placeholder tensor filled with zeros
            x_padded = torch.zeros((batch_size, in_c, padded_h, padded_w))

            # Copy the original image into the center of the padded tensor
            x_padded[:, :, self.padding : self.padding + h_in, self.padding : self.padding + w_in] = x
        else:
            # No padding needed, just use the original input
            x_padded = x

        # Calculate Output Size:
        # Using the standard convolution formula, and casting to int because dimensions must be integers
        h_out = int((h_in + 2 * self.padding - k_h) / self.stride + 1)
        w_out = int((w_in + 2 * self.padding - k_w) / self.stride + 1)

        # Initialize the output tensor
        output = torch.zeros((batch_size, self.out_channels, h_out, w_out))

        #Sliding Window Loops:
        # We iterate over the output dimensions
        for i in range(h_out):
            for j in range(w_out):

                # Calculating the starting and ending indices on the padded input
                # Stride determines the jump size
                h_start = i * self.stride
                h_end = h_start + k_h
                w_start = j * self.stride
                w_end = w_start + k_w

                # Extracting the relevant patch from the input
                patch = x_padded[:, :, h_start:h_end, w_start:w_end]


                # Now we need to multiply the patch by the weights
                # Using 'unsqueeze' for broadcasting so shapes align:
                res = patch.unsqueeze(1) * self.weight.unsqueeze(0)

                # Summing over the dimensions that "collapse" during convolution:
                val = res.sum(dim=(2, 3, 4))

                # Assigning the result to the correct pixel in the output
                output[:, :, i, j] = val

        return output

# Sanity Check
print("Running comparison check...")

# 1. Setup parameters
bs, in_c, out_c, h, w = 2, 3, 4, 10, 10
k_size = (3,3)
pad = 1
strd = 2

# 2. Create random dummy input
dummy_input = torch.randn(bs, in_c, h, w)

# 3. Initialize my layer
my_layer = MyConv2d(in_channels=in_c, out_channels=out_c, kernel_size=k_size, stride=strd, padding=pad)

# 4. Initialize PyTorch's layer
torch_layer = nn.Conv2d(in_channels=in_c, out_channels=out_c, kernel_size=k_size, stride=strd, padding=pad, bias=False)

# 5.Copy weights from my layer to PyTorch's layer
torch_layer.weight.data = my_layer.weight.data.clone()

# 6. Forward pass
my_output = my_layer(dummy_input)
torch_output = torch_layer(dummy_input)

# 7. Check the difference
diff = (my_output - torch_output).abs().max()
print(f"Difference between my implementation and PyTorch: {diff.item()}")

if diff < 1e-5:
    print("Yay! It works exactly the same.")
else:
    print("Oops, something is wrong.")
